# 01 · Audit of Z-FPV-v1 (owner: AK)

Five questions, answered from the labels only (no frames are shown or saved: dataset licence).

1. How big are the boxes: px, share of the frame, and px after letterbox to 640, per split?
2. Do val and test really differ from train in box size?
3. What share of frames is empty, per video?
4. Day / night per split.
5. Everything saved: tables in `results/audit/`, figures in `results/figures/`.

**Box size** = `sqrt(w * h)` in px. **Letterboxed** = after resizing the frame so its long side is 640,
which is what the model sees. The same box shrinks by a different factor per video
(1280×720 → ×0.5, 1920×1080 → ×0.33, portrait 576×1024 → ×0.625).

In [ ]:
import os
from pathlib import Path

# Run from the repo root, wherever the notebook was opened from.
while not Path("config/default.yaml").exists():
    os.chdir("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from zfpv.config import load_config
from zfpv.data.audit import (SIZE_LABELS, box_table, per_split_summary, per_video_summary,
                             size_shift_test, video_resolutions)
from zfpv.data.split import load_manifest
from zfpv.utils import set_seed

cfg = load_config()
set_seed(cfg["seed"])
SPLITS = cfg["split"]["names"]
RAW = Path(cfg["paths"]["raw_dir"])
TABLES = Path(cfg["paths"]["results_dir"]) / "audit"
FIGURES = Path(cfg["paths"]["results_dir"]) / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)

# Chart style: categorical slots 1-3 for splits (validated all-pairs), one blue ramp for
# ordered things (size buckets, day/night). Text never wears a series colour.
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
SPLIT_COLOR = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
BLUE_RAMP = ["#b7d3f6", "#6da7ec", "#256abf", "#0d366b"]  # light -> dark
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})

In [ ]:
manifest = load_manifest(Path(cfg["paths"]["split_manifest"]))
boxes = box_table(RAW, imgsz=640)
per_video = per_video_summary(boxes, manifest)   # raises if the manifest disagrees with labels
per_split = per_split_summary(boxes, per_video, SPLITS)

# The labels' image size must match the real video, or every box is in the wrong place.
res = video_resolutions(RAW).merge(per_video[["video", "img_w", "img_h"]], on="video")
mismatch = res[(res["video_w"] != res["img_w"]) | (res["video_h"] != res["img_h"])]
assert mismatch.empty, mismatch
print(f"{len(boxes)} boxes, {per_video['n_frames'].sum()} frames, {len(per_video)} videos; "
      "label sizes match the videos.")

## 1. Box size per split

In [ ]:
size_cols = ["split", "boxes", "size_px_med", "rel_area_med_pct",
             "size_lb_p10", "size_lb_med", "size_lb_p90", "video_median_of_size_lb_med",
             *[f"share_lb_{k}" for k in SIZE_LABELS]]
per_split[size_cols]

In [ ]:
# Per-video view: each video is one dot (its median letterboxed box size).
fig, ax = plt.subplots(figsize=(7, 4.2))
rng = np.random.default_rng(cfg["seed"])
for i, split in enumerate(SPLITS):
    v = per_video[per_video["split"] == split]
    x = i + rng.uniform(-0.12, 0.12, len(v))
    day, night = ~v["is_night"].to_numpy(), v["is_night"].to_numpy()
    ax.scatter(x[day], v["size_lb_med"][day], s=64, color=SPLIT_COLOR[split],
               edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.scatter(x[night], v["size_lb_med"][night], s=64, facecolor=SURFACE,
               edgecolor=SPLIT_COLOR[split], linewidth=2, zorder=3)
    med = v["size_lb_med"].median()
    ax.hlines(med, i - 0.28, i + 0.28, color=INK, linewidth=2, zorder=4)
    ax.annotate(f"median {med:.0f} px", (i + 0.3, med), va="center", fontsize=9, color=INK_2)
for ref in (8, 16, 32):
    ax.axhline(ref, color=GRID, linewidth=1, zorder=1)
ax.set_yscale("log")
ax.set_yticks([4, 8, 16, 32, 64, 128])
ax.set_yticklabels(["4", "8", "16", "32", "64", "128"])
ax.set_xticks(range(len(SPLITS)), SPLITS)
ax.set_xlim(-0.5, len(SPLITS) - 0.1)
ax.set_ylabel("median box size after letterbox to 640 (px, log)")
ax.set_title("Box size per video, by split")
ax.scatter([], [], s=64, color=INK_2, label="day video")
ax.scatter([], [], s=64, facecolor=SURFACE, edgecolor=INK_2, linewidth=2, label="night video")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "audit_box_size_by_split.png", dpi=200)

In [ ]:
# Share of boxes per letterboxed size bucket (frame-level: long videos weigh more).
fig, ax = plt.subplots(figsize=(7, 2.6))
left = np.zeros(len(SPLITS))
for k, color in zip(SIZE_LABELS, BLUE_RAMP):
    share = per_split.set_index("split").loc[SPLITS, f"share_lb_{k}"].to_numpy()
    ax.barh(SPLITS, share, left=left, color=color, edgecolor=SURFACE, linewidth=2,
            height=0.6, label=f"{k} px")
    for y, (l, s) in enumerate(zip(left, share)):
        if s >= 0.07:
            ax.text(l + s / 2, y, f"{s:.0%}", ha="center", va="center", fontsize=9,
                    color=INK if color in BLUE_RAMP[:2] else SURFACE)
    left += share
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Boxes by size after letterbox to 640")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18), fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "audit_size_buckets_by_split.png", dpi=200, bbox_inches="tight")

## 2. Do val and test really differ from train?

In [ ]:
shift = size_shift_test(per_video, col="size_lb_med")
shift

**Reading it.** The unit is the video (frames of one video are near-copies), so each video gives one
number. Two different statements:

- **Descriptively, yes: this split is uneven.** Three of the four val videos have median boxes around
  8 px after letterbox, and test sits between val and train. Checkpoints chosen on val will be chosen
  for tiny-object performance, and train has comparatively few tiny boxes.
- **Statistically, not established.** With 12 vs 4 videos, the exact permutation test gives
  p ≈ 0.13 (val) and p ≈ 0.28 (test): a gap this large happens often by chance when 20 very different
  videos are split at random. So it is not evidence of a biased split procedure, but it *is* the split
  every result will be measured on.

**Consequence.** Report every metric by size bucket, not only overall. And at 640 a typical val box is
≈ 8 px, i.e. one cell of YOLO's finest (stride-8) grid: try `imgsz=1280`, SAHI or a P2 head.

## 3. Empty frames per video

In [ ]:
per_video[["video", "split", "is_night", "n_frames", "n_annotated", "n_empty",
           "empty_share", "multi_box_frames"]].sort_values("empty_share", ascending=False)

In [ ]:
order = per_video.sort_values(["split", "empty_share"], key=lambda s: s.map(
    {k: i for i, k in enumerate(SPLITS)}) if s.name == "split" else s)
fig, ax = plt.subplots(figsize=(7, 5.6))
ax.barh(order["video"], order["empty_share"],
        color=[SPLIT_COLOR[s] for s in order["split"]], height=0.7)
for y, (share, n) in enumerate(zip(order["empty_share"], order["n_empty"])):
    label = f"{share:.0%} ({n})" if n else "0%"
    ax.text(share + 0.004, y, label, va="center", fontsize=8, color=INK_2)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.grid(axis="x")
ax.set_axisbelow(True)
ax.set_xlabel("share of frames without a drone")
ax.set_title("Empty frames per video")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=SPLIT_COLOR[s], label=s) for s in SPLITS], loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES / "audit_empty_share_by_video.png", dpi=200)

## 4. Day / night per split

In [ ]:
per_split[["split", "videos", "night_videos", "frames", "night_frames", "empty_share"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.6))
ps = per_split.set_index("split").loc[SPLITS]
day = ps["frames"] - ps["night_frames"]
ax.barh(SPLITS, day, color=BLUE_RAMP[1], edgecolor=SURFACE, linewidth=2, height=0.6, label="day")
ax.barh(SPLITS, ps["night_frames"], left=day, color=BLUE_RAMP[3], edgecolor=SURFACE,
        linewidth=2, height=0.6, label="night")
for y, split in enumerate(SPLITS):
    r = ps.loc[split]
    ax.text(r["frames"] + 40, y, f"{r['night_frames'] / r['frames']:.0%} night "
            f"({int(r['night_videos'])} of {int(r['videos'])} videos)",
            va="center", fontsize=9, color=INK_2)
ax.invert_yaxis()
ax.set_xlim(0, ps["frames"].max() * 1.45)
ax.set_xlabel("frames")
ax.set_title("Day and night frames per split")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES / "audit_day_night_by_split.png", dpi=200)

Each split has exactly one night video, so every night-specific number is a single-video number.

### 4b. Is `is_night` right? An objective check

`is_night` was set by hand in the manifest; no code computes it and nobody recorded how. Check it
against the frames: mean brightness $\bar V$ and saturation $\bar S$ (HSV, 0–255) over 20 evenly spaced
frames per video.

- **dark** ($\bar V$ < 60): low-light RGB, i.e. night.
- **monochrome** ($\bar S$ < 5): thermal or black-and-white, at *any* brightness. A thermal video at night
  is not necessarily dark, so $\bar V$ cannot judge it.
- **check** ($\bar S$ < 20): low colour; look at it.

These are suggestions for a human to confirm by eye, not labels.

In [ ]:
from zfpv.data.audit import CHECK_S, DARK_V, MONO_S, color_stats

color = color_stats(Path(cfg["paths"]["frames_dir"]), per_video["video"].tolist())
color = color.merge(per_video[["video", "split", "is_night"]], on="video")
color["agrees_with_manifest"] = color["suggest_dark"] == color["is_night"]
color.sort_values("mean_s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.axhline(DARK_V, color=GRID, linewidth=1, zorder=1)
ax.axvspan(0, MONO_S, color=GRID, alpha=0.6, linewidth=0, zorder=0)
ax.axvline(CHECK_S, color=GRID, linewidth=1, zorder=1)
ax.annotate(f"dark: V < {DARK_V}", (175, DARK_V - 3), ha="right", va="top", fontsize=8, color=INK_2)
ax.annotate("mono-\nchrome", (MONO_S / 2, 150), ha="center", fontsize=8, color=INK_2)
ax.annotate(f"check: S < {CHECK_S}", (CHECK_S + 2, 150), fontsize=8, color=INK_2)
for split in SPLITS:
    c = color[color["split"] == split]
    day, night = c[~c["is_night"]], c[c["is_night"]]
    ax.scatter(day["mean_s"], day["mean_v"], s=64, color=SPLIT_COLOR[split], edgecolor=SURFACE,
               linewidth=2, zorder=3, label=split)
    ax.scatter(night["mean_s"], night["mean_v"], s=64, facecolor=SURFACE,
               edgecolor=SPLIT_COLOR[split], linewidth=2, zorder=3)
nudge = {"video-8": (8, 7), "video-24": (8, -7)}  # these two sit almost on top of each other
for _, r in color[color["mean_s"] < CHECK_S].iterrows():  # label only the ones to look at
    ax.annotate(r["video"], (r["mean_s"], r["mean_v"]), xytext=nudge.get(r["video"], (8, 0)),
                textcoords="offset points", va="center", fontsize=8, color=INK_2)
ax.scatter([], [], s=64, facecolor=SURFACE, edgecolor=INK_2, linewidth=2, label="night (manifest)")
ax.set_xlim(0, 180)
ax.set_ylim(0, 160)
ax.set_xlabel("mean saturation S (HSV, 0-255)")
ax.set_ylabel("mean brightness V (HSV, 0-255)")
ax.set_title("Brightness and colour per video")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES / "audit_color_by_video.png", dpi=200)

**Reading it.**
- The three `is_night` videos (8, 24, 25) are the three darkest, with clear room to the next one:
  the brightness check **agrees with the manifest** on all 20 videos.
- **`video-9` is almost grey** ($\bar S$ ≈ 2) but not dark: most likely thermal or black-and-white. Whether
  it is day or night cannot be read from brightness.
- **`video-7`, `video-26`** and the night videos have low colour: look at them.
- Next: confirm by eye, then add a `modality` column (rgb / thermal) to the manifest and record in
  `data/README.md` who decided and how.

## 5. Save the tables

In [ ]:
per_video.to_csv(TABLES / "per_video.csv", index=False, float_format="%.4f")
per_split.to_csv(TABLES / "per_split.csv", index=False, float_format="%.4f")
shift.to_csv(TABLES / "size_shift_test.csv", index=False, float_format="%.4f")
color.to_csv(TABLES / "color_stats.csv", index=False, float_format="%.2f")
sorted(str(p) for p in [*TABLES.glob("*.csv"), *FIGURES.glob("audit_*.png")])